# Natural Language Processing - Assignment 1
## Track A: Short Answer Questions (SAQ)
### Cross-Cultural Knowledge Evaluation

**Student Name**: (John) Paul Nagle  
**Student ID**: R00065426  
**Model**: Mistral-7B-Instruct-v0.2  
**Locales**: ga-IE (Irish), en-US (English-US), ar-SA (Arabic-Saudi Arabia), zh-CN (Chinese-China)

## Setup and Installation

In [51]:
# Install required packages
!pip install -r requirements.txt -q

## Imports and Configuration

In [52]:
import warnings
import re
import unicodedata
import json
import random
import numpy as np
from typing import Dict,  Optional, Literal

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0
CUDA available: False


## Step 1: Locale Configuration

Selected locales meet assignment requirements:
- **en-US** (English-US): High-resource baseline
- **en-GB** (English-UK): High-resource, European locale
- **zh-CN** (Chinese-China): Non-Latin script, major language
- **am-ET** (Amharic-Ethiopia): Low-resource, under-represented locale

In [53]:
# Define locales
LOCALES = {
    'en-US': {
        'code': 'en-US',
        'name': 'English (United States)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'en-GB': {
        'code': 'en-GB',
        'name': 'English (United Kingdom)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'zh-CN': {
        'code': 'zh-CN',
        'name': 'Chinese (China)',
        'language': 'Simplified Chinese',
        'script': 'Han',
        'resource_level': 'high'
    },
    'am-ET': {
        'code': 'am-ET',
        'name': 'Amharic (Ethiopia)',
        'language': 'Amharic',
        'script': 'Ethiopic',
        'resource_level': 'low'
    }
}

print("Configured Locales:")
for locale_code, config in LOCALES.items():
    print(f"  {locale_code}: {config['name']} ({config['script']} script, {config['resource_level']}-resource)")

Configured Locales:
  en-US: English (United States) (Latin script, high-resource)
  en-GB: English (United Kingdom) (Latin script, high-resource)
  zh-CN: Chinese (China) (Han script, high-resource)
  am-ET: Amharic (Ethiopia) (Ethiopic script, low-resource)


## Step 2: Load Mistral-7B Model

**Hardware Detection & Model Loading Strategy:**
- GPU available: Use 4-bit quantization with Mistral-7B
- CPU only: Use smaller model (TinyLlama-1.1B) for faster inference

**Note**: For CPU-only systems, TinyLlama is recommended. For production with GPU, use Mistral-7B.

In [54]:
TEMPERATURE = 0.0  # Required for reproducibility
MAX_NEW_TOKENS = 100 # Maximum number of tokens to generate ????

# Configure model loading based on hardware
if torch.cuda.is_available():
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # 7B params
    # GPU: Use 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✓ Model {MODEL_NAME} loaded with 4-bit quantization on GPU")
    
else:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 1.1B params, faster on CPU
    # No quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    print("✓ Model loaded on CPU (float32)")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.eval()
print(f"✓ Model: {MODEL_NAME}")
print(f"✓ Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"✓ Temperature: {TEMPERATURE} (deterministic)")
print(f"✓ Max new tokens: {MAX_NEW_TOKENS}")

Loading weights: 100%|██████████| 201/201 [00:03<00:00, 50.34it/s]


✓ Model loaded on CPU (float32)
✓ Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
✓ Device: CPU
✓ Temperature: 0.0 (deterministic)
✓ Max new tokens: 100


## Step 3: Text Normalization

Robust normalization strategy for multilingual text matching.

In [55]:
class TextNormalizer:
    """Multi-stage text normalization for answer matching."""

    def __init__(self):
        self.normalization_form: Literal['NFC', 'NFD', 'NFKC', 'NFKD'] = 'NFC'  # Unicode normalization form

    def normalize(self, text: str, locale: Optional[str] = None) -> str:
        """Normalize text for multilingual text matching."""
        if not text:
            return ""

        # 1. Unicode normalization (NFC - Canonical Composition)
        text = unicodedata.normalize(self.normalization_form, text)

        # 2. Remove control characters
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]', '', text)

        # 3. Normalize line breaks
        text = text.replace('\r\n', '\n').replace('\r', '\n')

        # 4. Normalize whitespace (but preserve single spaces)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = text.strip()

        # 5. Convert all to lower case
        text = text.lower()

        # 6. Remove punctuation (but keep apostrophes for contractions)
        text = re.sub(r'[^\w\s\'\-]', '', text)

        return text

# Initialize normalizer
normalizer = TextNormalizer()


## Step 4: Baseline SAQ System

Direct prompting baseline with locale-aware generation.

In [56]:
class BaselineSAQSystem:
    """Baseline Short Answer Question system using direct prompting."""
    def __init__(self, model, tokenizer, normalizer, temperature=0.0):
        self.model = model
        self.tokenizer = tokenizer
        self.normalizer = normalizer
        self.temperature = temperature
        self.max_new_tokens = 100
    
    def create_prompt(self, question: str, locale: str) -> str:
        """ Create a prompt for the model. """
        locale_config = LOCALES.get(locale)
        if not locale_config:
            raise ValueError(f"Unknown locale: {locale}")
        
        # TinyLlama chat format
        if "TinyLlama" in MODEL_NAME:
            prompt = f"""<|system|>
                        You are a helpful assistant that answers questions about {locale_config['name']} culture.
                        Answer in {locale_config['language']} language only.</s>
                        <|user|>
                        {question}</s>
                        <|assistant|>
                        """
        else:
            # Mistral/Llama format
            prompt = f"""[INST] Answer the following question about {locale_config['name']} culture and everyday knowledge.
                        Provide a short, direct answer in {locale_config['language']}.

                        Question: {question}

                        Answer: [/INST]
                      """
    
        return prompt
    
    def generate_answer(self, question: str, locale: str) -> Dict:
        """ Generate answer for a question in the specified locale. """
        # Normalize question
        question = self.normalizer.normalize(question)
        
        # Create prompt
        prompt = self.create_prompt(question, locale)
        
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.model.device)
        
        # Generate with temperature=0 for reproducibility
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                max_length=None, 
                temperature=0.0,
                do_sample=False,  # Greedy decoding when temperature=0
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        # Decode only the new tokens (not the prompt)
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        # Normalize answer
        answer = self.normalizer.normalize(answer)
        
        return {
            'question': question,
            'locale': locale,
            'answer': answer,
            'raw_output': answer,
            'prompt': prompt
        }

# Initialize baseline system
baseline_system = BaselineSAQSystem(
    model=model,
    tokenizer=tokenizer,
    normalizer=normalizer,
    temperature=TEMPERATURE
)

print("✓ Baseline SAQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Locales: {', '.join(LOCALES.keys())}")

✓ Baseline SAQ System initialized
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Temperature: 0.0
  Locales: en-US, en-GB, zh-CN, am-ET


In [57]:
def evaluate_answer(generated_answer: str, question_id: str, 
                    reference_data: dict, normalizer) -> bool:
    """
    Check if generated answer matches any reference answer.
    
    Args:
        generated_answer: Model's generated answer (already normalized)
        question_id: Question ID (e.g., "Al-en-01")
        reference_data: Loaded JSON data from answers file
        normalizer: TextNormalizer instance
    
    Returns:
        bool: True if answer matches any reference answer
    """
    if question_id not in reference_data:
        return False
    
    # Get all acceptable answers for this question
    annotations = reference_data[question_id]['annotations']
    
    # Normalize generated answer
    normalized_generated = normalizer.normalize(generated_answer)
    
    # Check against all acceptable answers
    for annotation in annotations:
        for reference_answer in annotation['answers']:
            normalized_reference = normalizer.normalize(reference_answer)
            
            # Exact match after normalization
            if normalized_generated == normalized_reference:
                return True
            
            # Substring match (generated contains reference or vice versa)
            if normalized_reference in normalized_generated or \
               normalized_generated in normalized_reference:
                return True
    
    return False

## Step 5: Load Reference Answers

Load annotated reference answers from JSON files for evaluation.

In [58]:
import json

# Load reference answers for each locale
def load_reference_answers(locale_code):
    """Load reference answers from JSON file for a locale"""
    locale_to_file = {
        'en-US': 'US_data.json',
        'en-GB': 'UK_data.json',
        'zh-CN': 'China_data.json',
        'am-ET': 'Ethiopia_data.json'
    }
    
    filename = locale_to_file.get(locale_code)
    if not filename:
        return {}
    
    try:
        with open(f'answers/{filename}', 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return {}

# Load all reference data
REFERENCE_ANSWERS = {
    locale: load_reference_answers(locale) 
    for locale in LOCALES.keys()
}

print("✓ Reference answers loaded")
for locale, data in REFERENCE_ANSWERS.items():
    print(f"  {locale}: {len(data)} questions")

✓ Reference answers loaded
  en-US: 500 questions
  en-GB: 500 questions
  zh-CN: 500 questions
  am-ET: 500 questions


## Step 6: Load Questions with IDs

Load questions along with their IDs for proper evaluation.

In [59]:
import pandas as pd

def load_questions_with_ids(locale_code, num_questions=10):
    """Load questions with their IDs from CSV file"""
    locale_to_file = {
        'en-US': 'US_questions.csv',
        'en-GB': 'UK_questions.csv',
        'zh-CN': 'China_questions.csv',
        'am-ET': 'Ethiopia_questions.csv'
    }
    
    filename = locale_to_file.get(locale_code)
    if not filename:
        return []
    
    try:
        df = pd.read_csv(f'questions/{filename}')
        # Return list of (question_id, question_text) tuples
        questions = [
            (row['ID'], row['Question']) 
            for _, row in df.head(num_questions).iterrows()
        ]
        return questions
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return []

# Test loading
test_questions = load_questions_with_ids('en-GB', 3)
print("✓ Question loading function ready")
print(f"  Sample: {test_questions[0] if test_questions else 'None'}")

✓ Question loading function ready
  Sample: ('Al-en-01', 'What is a common snack for nursery kids in the UK?')


## Step 7: Evaluate Baseline System

Run full evaluation across all locales and calculate accuracy metrics.

In [ ]:
# Evaluate baseline system
NUM_EVAL_QUESTIONS = 10  # Start with 10 questions per locale

results = {locale: {'correct': 0, 'total': 0, 'details': []} for locale in LOCALES.keys()}

print("Evaluating Baseline System:")
print(f"Testing {NUM_EVAL_QUESTIONS} questions per locale\n")
print("="*80)

for locale in LOCALES.keys():
    print(f"\n{'='*80}")
    print(f"Locale: {locale} ({LOCALES[locale]['name']})")
    print(f"{'='*80}\n")
    
    # Load questions with IDs
    questions_with_ids = load_questions_with_ids(locale, NUM_EVAL_QUESTIONS)
    
    for i, (question_id, question_text) in enumerate(questions_with_ids, 1):
        # Generate answer
        result = baseline_system.generate_answer(question_text, locale)
        generated_answer = result['answer']
        
        # Evaluate answer
        is_correct = evaluate_answer(
            generated_answer, 
            question_id, 
            REFERENCE_ANSWERS[locale],
            normalizer
        )
        
        # Update results
        results[locale]['total'] += 1
        if is_correct:
            results[locale]['correct'] += 1
        
        # Store details
        results[locale]['details'].append({
            'question_id': question_id,
            'question': question_text,
            'generated': generated_answer,
            'correct': is_correct
        })
        
        # Print result
        status = "✓" if is_correct else "✗"
        print(f"{i}. {status} [{question_id}]")
        print(f"   Q: {question_text[:70]}..." if len(question_text) > 70 else f"   Q: {question_text}")
        print(f"   A: {generated_answer[:70]}..." if len(generated_answer) > 70 else f"   A: {generated_answer}")
        print("-"*80)

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")

Evaluating Baseline System:
Testing 10 questions per locale


Locale: en-US (English (United States))

1. ✓ [Al-en-01]
   Q: What is a common snack for preschool kids in the US?
   A: 1 cheerios 2 rice cakes with peanut butter 3 apple slices with almond ...
--------------------------------------------------------------------------------
2. ✓ [Al-en-02]
   Q: What is a popular food to go with beer in the US?
   A: 1 pizza 2 nachos 3 fries 4 burritos 5 tacos 6 grilled cheese sandwich ...
--------------------------------------------------------------------------------
3. ✓ [Al-en-04]
   Q: What is the most popular fruit in the US?
   A: 1 apple 2 banana 3 oranges 4 peaches 5 pineapples 6 mango 7 kiwi 8 str...
--------------------------------------------------------------------------------
4. ✓ [Al-en-06]
   Q: What is a common school cafeteria food in the US?
   A: 1 chicken nuggets 2 french fries 3 pizza 4 burrito 5 mac and cheese 6 ...
---------------------------------------------------

## Step 8: Display Results Summary

Show accuracy metrics per locale and overall performance.

In [ ]:
# Print summary
print(f"\n{'='*80}")
print("EVALUATION SUMMARY")
print(f"{'='*80}\n")

overall_correct = sum(r['correct'] for r in results.values())
overall_total = sum(r['total'] for r in results.values())
overall_accuracy = (overall_correct / overall_total * 100) if overall_total > 0 else 0

print(f"{'Locale':<10} {'Correct':<10} {'Total':<10} {'Accuracy':<10}")
print("-"*80)

for locale in LOCALES.keys():
    correct = results[locale]['correct']
    total = results[locale]['total']
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"{locale:<10} {correct:<10} {total:<10} {accuracy:>6.1f}%")

print("-"*80)
print(f"{'Overall':<10} {overall_correct:<10} {overall_total:<10} {overall_accuracy:>6.1f}%")
print("="*80)


EVALUATION SUMMARY

Locale     Correct    Total      Accuracy  
--------------------------------------------------------------------------------
en-US      9          10           90.0%
en-GB      8          10           80.0%
zh-CN      1          10           10.0%
am-ET      0          10            0.0%
--------------------------------------------------------------------------------
Overall    18         40           45.0%


## Step 9: Analyze Failures

Examine incorrect answers to understand model weaknesses.

In [ ]:
# Show failure examples
print("\n" + "="*80)
print("FAILURE ANALYSIS")
print("="*80 + "\n")

total_failures = 0

for locale in LOCALES.keys():
    failures = [d for d in results[locale]['details'] if not d['correct']]
    total_failures += len(failures)
    
    if failures:
        print(f"\n{locale} - {len(failures)} failures:")
        print("-"*80)
        
        for idx, fail in enumerate(failures[:5], 1):  # Show first 5 failures
            print(f"\n{idx}. ID: {fail['question_id']}")
            print(f"   Question: {fail['question']}")
            print(f"   Generated: {fail['generated']}")
            
            # Show expected answers
            if fail['question_id'] in REFERENCE_ANSWERS[locale]:
                expected = REFERENCE_ANSWERS[locale][fail['question_id']]['annotations']
                expected_list = [a['answers'][0] for a in expected[:3]]
                print(f"   Expected: {', '.join(expected_list)}")

print(f"\n{'='*80}")
print(f"Total failures across all locales: {total_failures}/{overall_total}")
print(f"{'='*80}")


FAILURE ANALYSIS


en-US - 1 failures:
--------------------------------------------------------------------------------

1. ID: Al-en-02
   Question: What is a popular food to go with beer in the US?
   Generated: a popular food to go with beer in the united states is a classic american appetizer called beer cheese this is a cheese that has been melted and spread on a slice of bread usually with a side of pickles or other condiments it is often served with a glass of beer or a bottle of wine beer cheese is a delicious and easy-to-make appetizer that pairs well with a variety of beers
   Expected: nuts, pretzels, barbeque

en-GB - 2 failures:
--------------------------------------------------------------------------------

1. ID: Al-en-09
   Question: What is a popular snack at an amusement park in the UK?
   Generated: a popular snack at an amusement park in the uk is a hot dog hot dogs are a popular food item at amusement parks especially in the uk where they are known as dogs hot do